# PySpark Joins — single notebook examples

This notebook contains steps **00–10** with small realistic datasets.

Dataset theme was changed from generic `customers/orders/products` to a **streaming platform** domain:

- `users` — registered users
- `subscriptions` — current user subscription plan
- `watch_events` — viewing activity
- `devices` — device inventory per user
- `content_catalog` — movies and shows
- `campaign_signups` — marketing campaign signups
- `support_tickets` — support activity

Target environment: local Apache Spark cluster, e.g. `spark://spark-master:7077` from Docker Compose.


## 00 — Spark setup and sample data


In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("pyspark-joins-single-notebook")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.version


'4.0.2'

In [2]:
users = spark.createDataFrame([
    (1, "Anna", "SK", "2024-01-10"),
    (2, "Boris", "CZ", "2024-01-15"),
    (3, "Cyril", "SK", "2024-02-01"),
    (4, "Dana", "AT", "2024-02-20"),
    (5, "Eva", "SK", "2024-03-05"),
    (6, "Filip", "DE", "2024-03-11"),
    (7, "Gina", None, "2024-03-18"),
], ["user_id", "name", "country_code", "registered_at"])

subscriptions = spark.createDataFrame([
    (1, "premium", "active"),
    (2, "basic", "active"),
    (3, "premium", "cancelled"),
    (5, "family", "active"),
    (8, "basic", "active"),
], ["user_id", "plan", "status"])

watch_events = spark.createDataFrame([
    (1001, 1, "c01", "2024-04-01", 35),
    (1002, 1, "c02", "2024-04-01", 20),
    (1003, 2, "c01", "2024-04-02", 10),
    (1004, 2, "c03", "2024-04-02", 55),
    (1005, 3, "c04", "2024-04-03", 12),
    (1006, 5, "c01", "2024-04-04", 40),
    (1007, 5, "c05", "2024-04-05", 18),
    (1008, 8, "c02", "2024-04-05", 22),
    (1009, None, "c06", "2024-04-06", 5),
], ["event_id", "user_id", "content_id", "event_date", "minutes_watched"])

content_catalog = spark.createDataFrame([
    ("c01", "Galaxy Roads", "movie", "sci-fi"),
    ("c02", "Data Detectives", "series", "crime"),
    ("c03", "Cooking Fast", "series", "lifestyle"),
    ("c04", "The Lake", "movie", "drama"),
    ("c05", "Spark Academy", "series", "education"),
], ["content_id", "title", "content_type", "genre"])

devices = spark.createDataFrame([
    (1, "tv", "living-room"),
    (1, "mobile", "anna-phone"),
    (2, "tablet", "boris-ipad"),
    (3, "tv", "bedroom"),
    (5, "mobile", "eva-phone"),
    (5, "tv", "kids-room"),
    (7, "mobile", "gina-phone"),
], ["user_id", "device_type", "device_name"])

campaign_signups = spark.createDataFrame([
    ("spring-2024", 1, "SK"),
    ("spring-2024", 2, "CZ"),
    ("spring-2024", 4, "AT"),
    ("summer-2024", 5, "SK"),
    ("summer-2024", 8, "HU"),
    ("summer-2024", None, "SK"),
], ["campaign_id", "user_id", "country_code"])

support_tickets = spark.createDataFrame([
    (501, 1, "billing", "closed"),
    (502, 2, "playback", "open"),
    (503, 2, "login", "closed"),
    (504, 6, "billing", "open"),
    (505, None, "unknown", "open"),
], ["ticket_id", "user_id", "category", "ticket_status"])

for name, df in {
    "users": users,
    "subscriptions": subscriptions,
    "watch_events": watch_events,
    "content_catalog": content_catalog,
    "devices": devices,
    "campaign_signups": campaign_signups,
    "support_tickets": support_tickets,
}.items():
    df.createOrReplaceTempView(name)
    print(name)
    df.show(truncate=False)


users


+-------+-----+------------+-------------+
|user_id|name |country_code|registered_at|
+-------+-----+------------+-------------+
|1      |Anna |SK          |2024-01-10   |
|2      |Boris|CZ          |2024-01-15   |
|3      |Cyril|SK          |2024-02-01   |
|4      |Dana |AT          |2024-02-20   |
|5      |Eva  |SK          |2024-03-05   |
|6      |Filip|DE          |2024-03-11   |
|7      |Gina |NULL        |2024-03-18   |
+-------+-----+------------+-------------+

subscriptions
+-------+-------+---------+
|user_id|plan   |status   |
+-------+-------+---------+
|1      |premium|active   |
|2      |basic  |active   |
|3      |premium|cancelled|
|5      |family |active   |
|8      |basic  |active   |
+-------+-------+---------+

watch_events
+--------+-------+----------+----------+---------------+
|event_id|user_id|content_id|event_date|minutes_watched|
+--------+-------+----------+----------+---------------+
|1001    |1      |c01       |2024-04-01|35             |
|1002    |1      |

## 01 — Inner join

Users with a subscription. Rows without matching keys on both sides disappear.


In [3]:
users.join(subscriptions, on="user_id", how="inner").show(truncate=False)


+-------+-----+------------+-------------+-------+---------+
|user_id|name |country_code|registered_at|plan   |status   |
+-------+-----+------------+-------------+-------+---------+
|2      |Boris|CZ          |2024-01-15   |basic  |active   |
|1      |Anna |SK          |2024-01-10   |premium|active   |
|5      |Eva  |SK          |2024-03-05   |family |active   |
|3      |Cyril|SK          |2024-02-01   |premium|cancelled|
+-------+-----+------------+-------------+-------+---------+



## 02 — Left, right and full outer joins

Useful for keeping unmatched records from one or both sides.


In [4]:
print("LEFT JOIN: all users, subscription if available")
users.join(subscriptions, on="user_id", how="left").orderBy("user_id").show(truncate=False)

print("RIGHT JOIN: all subscriptions, user profile if available")
users.join(subscriptions, on="user_id", how="right").orderBy("user_id").show(truncate=False)

print("FULL JOIN: everything from both sides")
users.join(subscriptions, on="user_id", how="full").orderBy("user_id").show(truncate=False)


LEFT JOIN: all users, subscription if available
+-------+-----+------------+-------------+-------+---------+
|user_id|name |country_code|registered_at|plan   |status   |
+-------+-----+------------+-------------+-------+---------+
|1      |Anna |SK          |2024-01-10   |premium|active   |
|2      |Boris|CZ          |2024-01-15   |basic  |active   |
|3      |Cyril|SK          |2024-02-01   |premium|cancelled|
|4      |Dana |AT          |2024-02-20   |NULL   |NULL     |
|5      |Eva  |SK          |2024-03-05   |family |active   |
|6      |Filip|DE          |2024-03-11   |NULL   |NULL     |
|7      |Gina |NULL        |2024-03-18   |NULL   |NULL     |
+-------+-----+------------+-------------+-------+---------+

RIGHT JOIN: all subscriptions, user profile if available
+-------+-----+------------+-------------+-------+---------+
|user_id|name |country_code|registered_at|plan   |status   |
+-------+-----+------------+-------------+-------+---------+
|1      |Anna |SK          |2024-01-10  

## 03 — Left semi and left anti joins

- `left_semi`: return left rows that have a match on the right
- `left_anti`: return left rows that do not have a match on the right


In [5]:
print("Users who have watched something")
users.join(watch_events.select("user_id").distinct(), on="user_id", how="left_semi").show(truncate=False)

print("Users with no watch activity")
users.join(watch_events.select("user_id").distinct(), on="user_id", how="left_anti").show(truncate=False)


Users who have watched something
+-------+-----+------------+-------------+
|user_id|name |country_code|registered_at|
+-------+-----+------------+-------------+
|2      |Boris|CZ          |2024-01-15   |
|1      |Anna |SK          |2024-01-10   |
|3      |Cyril|SK          |2024-02-01   |
|5      |Eva  |SK          |2024-03-05   |
+-------+-----+------------+-------------+

Users with no watch activity
+-------+-----+------------+-------------+
|user_id|name |country_code|registered_at|
+-------+-----+------------+-------------+
|4      |Dana |AT          |2024-02-20   |
|6      |Filip|DE          |2024-03-11   |
|7      |Gina |NULL        |2024-03-18   |
+-------+-----+------------+-------------+



## 04 — Cross join and self join

Use cross joins carefully. They multiply rows: `left_count * right_count`.


In [6]:
small_plans = spark.createDataFrame([
    ("basic",),
    ("premium",),
    ("family",),
], ["plan"])

small_countries = spark.createDataFrame([
    ("SK",),
    ("CZ",),
], ["country_code"])

small_plans.crossJoin(small_countries).show(truncate=False)


+-------+------------+
|plan   |country_code|
+-------+------------+
|basic  |SK          |
|basic  |CZ          |
|premium|SK          |
|family |SK          |
|premium|CZ          |
|family |CZ          |
+-------+------------+



In [7]:
print("Self join: users from the same country")
left_users = users.alias("left")
right_users = users.alias("right")

(
    left_users
    .join(
        right_users,
        (F.col("left.country_code") == F.col("right.country_code")) &
        (F.col("left.user_id") < F.col("right.user_id")),
        "inner"
    )
    .select(
        F.col("left.name").alias("user_a"),
        F.col("right.name").alias("user_b"),
        F.col("left.country_code")
    )
    .show(truncate=False)
)


Self join: users from the same country
+------+------+------------+
|user_a|user_b|country_code|
+------+------+------------+
|Anna  |Eva   |SK          |
|Anna  |Cyril |SK          |
|Cyril |Eva   |SK          |
+------+------+------------+



## 05 — Multi-column joins

Join marketing signups to users by both `user_id` and `country_code`.


In [8]:
(
    campaign_signups.alias("c")
    .join(
        users.alias("u"),
        (F.col("c.user_id") == F.col("u.user_id")) &
        (F.col("c.country_code") == F.col("u.country_code")),
        "left"
    )
    .select("c.campaign_id", "c.user_id", "c.country_code", "u.name")
    .orderBy("campaign_id", "user_id")
    .show(truncate=False)
)


+-----------+-------+------------+-----+
|campaign_id|user_id|country_code|name |
+-----------+-------+------------+-----+
|spring-2024|1      |SK          |Anna |
|spring-2024|2      |CZ          |Boris|
|spring-2024|4      |AT          |Dana |
|summer-2024|NULL   |SK          |NULL |
|summer-2024|5      |SK          |Eva  |
|summer-2024|8      |HU          |NULL |
+-----------+-------+------------+-----+



## 06 — Nulls and duplicate keys

Normal equality does not match `NULL = NULL`. Use null-safe equality `<=>` in SQL or `eqNullSafe` in the DataFrame API.


In [9]:
print("Normal join: NULL keys do not match")
(
    watch_events.alias("w")
    .join(support_tickets.alias("t"), F.col("w.user_id") == F.col("t.user_id"), "inner")
    .select("w.event_id", "w.user_id", "t.ticket_id", "t.category")
    .show(truncate=False)
)

print("Null-safe join: NULL keys can match")
(
    watch_events.alias("w")
    .join(support_tickets.alias("t"), F.col("w.user_id").eqNullSafe(F.col("t.user_id")), "inner")
    .select("w.event_id", "w.user_id", "t.ticket_id", "t.category")
    .show(truncate=False)
)


Normal join: NULL keys do not match
+--------+-------+---------+--------+
|event_id|user_id|ticket_id|category|
+--------+-------+---------+--------+
|1004    |2      |502      |playback|
|1003    |2      |502      |playback|
|1002    |1      |501      |billing |
|1001    |1      |501      |billing |
|1004    |2      |503      |login   |
|1003    |2      |503      |login   |
+--------+-------+---------+--------+

Null-safe join: NULL keys can match
+--------+-------+---------+--------+
|event_id|user_id|ticket_id|category|
+--------+-------+---------+--------+
|1001    |1      |501      |billing |
|1002    |1      |501      |billing |
|1003    |2      |503      |login   |
|1003    |2      |502      |playback|
|1004    |2      |503      |login   |
|1004    |2      |502      |playback|
|1009    |NULL   |505      |unknown |
+--------+-------+---------+--------+



In [10]:
print("Duplicate keys multiply rows")
(
    users
    .join(devices, on="user_id", how="inner")
    .select("user_id", "name", "device_type", "device_name")
    .orderBy("user_id", "device_type")
    .show(truncate=False)
)


Duplicate keys multiply rows
+-------+-----+-----------+-----------+
|user_id|name |device_type|device_name|
+-------+-----+-----------+-----------+
|1      |Anna |mobile     |anna-phone |
|1      |Anna |tv         |living-room|
|2      |Boris|tablet     |boris-ipad |
|3      |Cyril|tv         |bedroom    |
|5      |Eva  |mobile     |eva-phone  |
|5      |Eva  |tv         |kids-room  |
|7      |Gina |mobile     |gina-phone |
+-------+-----+-----------+-----------+



## 07 — Broadcast join

Broadcast small dimension tables such as `content_catalog` to avoid shuffling the large fact table.


In [11]:
joined = (
    watch_events
    .join(F.broadcast(content_catalog), on="content_id", how="left")
    .select("event_id", "user_id", "title", "genre", "minutes_watched")
)

joined.show(truncate=False)
joined.explain(mode="formatted")


+--------+-------+---------------+---------+---------------+
|event_id|user_id|title          |genre    |minutes_watched|
+--------+-------+---------------+---------+---------------+
|1001    |1      |Galaxy Roads   |sci-fi   |35             |
|1002    |1      |Data Detectives|crime    |20             |
|1003    |2      |Galaxy Roads   |sci-fi   |10             |
|1004    |2      |Cooking Fast   |lifestyle|55             |
|1005    |3      |The Lake       |drama    |12             |
|1006    |5      |Galaxy Roads   |sci-fi   |40             |
|1007    |5      |Spark Academy  |education|18             |
|1008    |8      |Data Detectives|crime    |22             |
|1009    |NULL   |NULL           |NULL     |5              |
+--------+-------+---------------+---------+---------------+

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- BroadcastHashJoin LeftOuter BuildRight (7)
      :- Project (2)
      :  +- Scan ExistingRDD (1)
      +- BroadcastExchange (6)
         +- Pro

## 08 — Join strategies and explain plan

Compare physical plans with and without a broadcast hint.


In [12]:
print("Without explicit broadcast hint")
watch_events.join(content_catalog, on="content_id", how="inner").explain(mode="formatted")

print("With explicit broadcast hint")
watch_events.join(F.broadcast(content_catalog), on="content_id", how="inner").explain(mode="formatted")


Without explicit broadcast hint
== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan ExistingRDD (1)
      +- Sort (8)
         +- Exchange (7)
            +- Filter (6)
               +- Scan ExistingRDD (5)


(1) Scan ExistingRDD
Output [5]: [event_id#7L, user_id#8L, content_id#9, event_date#10, minutes_watched#11L]
Arguments: [event_id#7L, user_id#8L, content_id#9, event_date#10, minutes_watched#11L], MapPartitionsRDD[14] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [5]: [event_id#7L, user_id#8L, content_id#9, event_date#10, minutes_watched#11L]
Condition : isnotnull(content_id#9)

(3) Exchange
Input [5]: [event_id#7L, user_id#8L, content_id#9, event_date#10, minutes_watched#11L]
Arguments: hashpartitioning(content_id#9, 4), ENSURE_REQUIREMENTS, [plan_id=1702]

(4) Sort
Input [5]: [event

## 09 — Skewed join problem

This small example simulates one very frequent key. In real data, skew can cause one task to process much more data than others.


In [13]:
skewed_events = spark.createDataFrame(
    [(i, 1, "c01", 1) for i in range(1, 101)] +
    [(101, 2, "c02", 1), (102, 3, "c03", 1), (103, 5, "c04", 1)],
    ["event_id", "user_id", "content_id", "minutes_watched"]
)

print("Key distribution")
skewed_events.groupBy("user_id").count().orderBy(F.desc("count")).show()

print("Join result still correct, but key 1 is much heavier")
skewed_events.join(users, on="user_id", how="left").groupBy("user_id", "name").count().orderBy(F.desc("count")).show()


Key distribution
+-------+-----+
|user_id|count|
+-------+-----+
|      1|  100|
|      2|    1|
|      5|    1|
|      3|    1|
+-------+-----+

Join result still correct, but key 1 is much heavier
+-------+-----+-----+
|user_id| name|count|
+-------+-----+-----+
|      1| Anna|  100|
|      2|Boris|    1|
|      5|  Eva|    1|
|      3|Cyril|    1|
+-------+-----+-----+



In [14]:
print("Simple salting pattern for a skewed key")
SALT_BUCKETS = 4

salted_events = skewed_events.withColumn(
    "salt",
    F.when(F.col("user_id") == 1, (F.rand(seed=42) * SALT_BUCKETS).cast("int")).otherwise(F.lit(0))
)

# Spark does not allow explode() nested inside when()/otherwise().
# Build the conditional array first, then explode it as the top-level expression.
salt_array = F.when(
    F.col("user_id") == 1,
    F.array(*[F.lit(i) for i in range(SALT_BUCKETS)])
).otherwise(F.array(F.lit(0)))

salted_users = users.withColumn("salt", F.explode(salt_array))

(
    salted_events
    .join(salted_users, on=["user_id", "salt"], how="left")
    .groupBy("user_id", "name")
    .count()
    .orderBy(F.desc("count"))
    .show()
)


Simple salting pattern for a skewed key
+-------+-----+-----+
|user_id| name|count|
+-------+-----+-----+
|      1| Anna|  100|
|      3|Cyril|    1|
|      2|Boris|    1|
|      5|  Eva|    1|
+-------+-----+-----+



## 10 — Bucketed joins optional

Bucketed joins can reduce shuffle when both sides are saved bucketed by the join key and reused across workloads. This is optional and depends on table format, catalog, and Spark configuration.


In [15]:
spark.sql("DROP TABLE IF EXISTS bucketed_watch_events")
spark.sql("DROP TABLE IF EXISTS bucketed_users")

spark.conf.set("spark.sql.sources.bucketing.enabled", "true")

(
    watch_events
    .write
    .mode("overwrite")
    .bucketBy(4, "user_id")
    .sortBy("user_id")
    .saveAsTable("bucketed_watch_events")
)

(
    users
    .write
    .mode("overwrite")
    .bucketBy(4, "user_id")
    .sortBy("user_id")
    .saveAsTable("bucketed_users")
)

bucketed_join = spark.table("bucketed_watch_events").join(
    spark.table("bucketed_users"),
    on="user_id",
    how="inner"
)

bucketed_join.show(truncate=False)
bucketed_join.explain(mode="formatted")


+-------+--------+----------+----------+---------------+-----+------------+-------------+
|user_id|event_id|content_id|event_date|minutes_watched|name |country_code|registered_at|
+-------+--------+----------+----------+---------------+-----+------------+-------------+
|5      |1006    |c01       |2024-04-04|40             |Eva  |SK          |2024-03-05   |
|5      |1007    |c05       |2024-04-05|18             |Eva  |SK          |2024-03-05   |
|1      |1001    |c01       |2024-04-01|35             |Anna |SK          |2024-01-10   |
|1      |1002    |c02       |2024-04-01|20             |Anna |SK          |2024-01-10   |
|2      |1003    |c01       |2024-04-02|10             |Boris|CZ          |2024-01-15   |
|2      |1004    |c03       |2024-04-02|55             |Boris|CZ          |2024-01-15   |
|3      |1005    |c04       |2024-04-03|12             |Cyril|SK          |2024-02-01   |
+-------+--------+----------+----------+---------------+-----+------------+-------------+

== Physic

## Cleanup


In [16]:
# Optional cleanup
# spark.sql("DROP TABLE IF EXISTS bucketed_watch_events")
# spark.sql("DROP TABLE IF EXISTS bucketed_users")
# spark.stop()
